<h1 align="center">Automatización y flujos de trabajo con Python</h1>

# U4.01 – Estructura de proyecto de análisis

## Organización de carpetas: La Estructura Estándar 

```text

proyecto_etl/
├── data/
│   ├── raw/                   # Datos originales (sin modificar)
│   │   ├── ventas_enero.csv
│   │   ├── ventas_febrero.csv
│   │   └── productos.xlsx
│   └── processed/             # Datos después del pipeline (generados)
│       └── consolidado_final.csv
├── src/                       # Código reutilizable
│   └── utils.py               # Funciones auxiliares
├── notebooks/                 # Notebooks de análisis y exploración
│   └── Pipeline_ETL.ipynb
├── logs/                      # Archivos de log
│   └── pipeline.log
└── pipeline_etl.py                  # Script principal en Raíz

Creación de estructura desde la terminal
```bash
# Windows (Command Prompt)
mkdir proyecto_etl
cd proyecto_etl
mkdir data\raw data\processed src notebooks logs


# Linux/Mac (Terminal)
mkdir -p proyecto_etl/{data/raw,data/processed,src,notebooks,logs}
cd proyecto_etl
```

## Manejo Robusto de Rutas Relativas

¿Qué son las rutas relativas?

Son rutas que no dependen de la ubicación absoluta en tu computadora, sino que se basan en la posición relativa al archivo que las está usando.

## Buenas Prácticas de Nomenclatura

**Nombres de Archivos**

❌ MALAS PRÁCTICAS
- datos.csv                  # Muy genérico
- Ventas Final FINAL v2.csv  # Espacios, versiones confusas
- data-2023.csv              # Poco descriptivo

✅ BUENAS PRÁCTICAS
- ventas_enero_2024.csv      # Descriptivo, fecha clara
- clientes_activos.csv       # Propósito claro
- productos_categorias.xlsx  # Relación explícita

Convenciones:
- snake_case (guiones bajos)
- Minúsculas
- Descriptivo
- Fecha al final: YYYY-MM-DD o YYYYMMDD

**Nombres de Carpetas**

✅ ESTRUCTURA CLARA Y CONSISTENTE

```text
proyecto_etl/
├── data/                  # Todo relacionado con datos
│   ├── raw/              # Datos originales (no modificar)
│   ├── processed/        # Datos procesados (generar)
│   └── external/         # Datos de APIs o fuentes externas
├── src/                   # Código fuente (source)
├── notebooks/             # Notebooks de análisis
├── logs/                  # Archivos de log
├── tests/                 # Tests unitarios (opcional)
└── docs/                  # Documentación (opcional)
````

Convenciones:
- Nombres descriptivos en inglés
- snake_case
- Plural para colecciones (logs, tests)

**Nombres de Variables**

❌ MALAS PRÁCTICAS
- d = pd.read_csv('ventas.csv')     # Muy corto
- ventas_del_mes_de_enero = ...      # Muy largo
- VentasEnero = ...                   # CamelCase (Python usa snake_case)
- ventasEnero = ...                   # camelCase (no Python)


✅ BUENAS PRÁCTICAS
- df_ventas = pd.read_csv('ventas.csv')           # Claro y descriptivo
- df_ventas_enero = df_ventas[df_ventas['mes'] == 1]
- total_ventas = df_ventas['total'].sum()
- ganancia_promedio = df_ventas['ganancia'].mean()

Convenciones:
- snake_case
- Descriptivo pero conciso
- df_ para DataFrames
- lista_ para listas
- dict_ para diccionarios (opcional)

**Nombres de Funciones**

❌ MALAS PRÁCTICAS
- def f(x):           # No descriptivo
- def CalcularTotal(df):  # CamelCase (no Python)
- def calcular_el_total_de_ventas_del_dataframe(df):  # Muy largo


✅ BUENAS PRÁCTICAS

```python
def calcular_total_venta(df):
    """Calcula total de ventas multiplicando cantidad por precio."""
    return df['cantidad'] * df['precio_unitario']

def limpiar_espacios(df):
    """Elimina espacios en blanco de columnas texto."""
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
    return df

def validar_columnas_requeridas(df, columnas):
    """Verifica que el DataFrame tenga las columnas requeridas."""
    faltantes = set(columnas) - set(df.columns)
    if faltantes:
        raise ValueError(f"Columnas faltantes: {faltantes}")
    return True
```

Convenciones:
- snake_case
- Verbo + sustantivo (acción clara)
- Docstring explicando qué hace
- Nombres consistentes (calcular_, validar_, generar_)

**Nombres en Inglés vs Español**

Decisión de equipo: Ser consistente

```python
✅ OPCIÓN A: Todo en español

def calcular_total_ventas(df):
    return df['cantidad'] * df['precio_unitario']

✅ OPCIÓN B: Todo en inglés
def calculate_total_sales(df):
    return df['quantity'] * df['unit_price']

❌ MEZCLAR (inconsistente)
def calcular_total_sales(df):  # Mezcla español/inglés
    return df['cantidad'] * df['unit_price']

```
Recomendaciones:
- Código/funciones: inglés (estándar industria)
- Columnas de datos: español (más natural para dataset local)
- Documentación: español (tu audiencia)

# U4.02 - Funciones reutilizables

## Encapsular Pasos de Limpieza en Funciones Modulares

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import logging

# ============================================================================
# FUNCIÓN DE CARGA
# ============================================================================

def cargar_multiples_csv(directorio, patron='*.csv'):
    """
    Carga múltiples archivos CSV desde un directorio.
    
    Args:
        directorio (str): Ruta del directorio
        patron (str): Patrón de búsqueda (ej: 'ventas_*.csv')
        
    Returns:
        pd.DataFrame: Datos consolidados de todos los archivos
    """    
    archivos = glob.glob(os.path.join(directorio, patron))
    
    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos con patrón: {patron}")
    
    dfs = []
    for archivo in sorted(archivos):
        df = pd.read_csv(archivo)
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)


# ============================================================================
# FUNCIÓN DE TRANSFORMACIÓN
# ============================================================================

def limpiar_espacios(df):
    """
    Elimina espacios en blanco al inicio y final de columnas texto.
    
    Args:
        df (pd.DataFrame): DataFrame a limpiar
        
    Returns:
        pd.DataFrame: DataFrame limpio
    """
    df_limpio = df.copy()
    
    # Aplicar a columnas de tipo object (texto)
    for col in df_limpio.select_dtypes(include='object').columns:
        df_limpio[col] = df_limpio[col].str.strip()
    
    return df_limpio

# ============================================================================
# CONFIGURACIÓN DE LOGGING
# ============================================================================

def configurar_logging(nombre_archivo="pipeline.log"):
    """
    Configura el logging para registrar el flujo del pipeline.
    
    Args:
        nombre_archivo (str): Ruta del archivo de log
        
    Returns:
        logging.Logger: Logger configurado
    """ 
    
    base_dir = Path(__file__).resolve().parent.parent

    logs_dir = base_dir / "logs"
    logs_dir.mkdir(exist_ok=True)

    log_path = logs_dir / nombre_archivo

    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    # Evitar duplicados
    for h in logger.handlers[:]:
        h.close()
        logger.removeHandler(h)

    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(formatter)

    ch = logging.StreamHandler()
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    logger.addHandler(ch)

    return logger

## Crear archivo utils.py con funciones auxiliares

Se agregan las funciones a utils.py y se agrega la ruta a sys, para luego importarlas

## Documentación básica de funciones

```python
def nombre_funcion(arg1, arg2, arg3=valor_por_defecto):
    """
    Descripción breve de una línea de lo que hace.
    
    Descripción más amplia si es necesario. Explica el contexto,
    por qué esta función es útil, y cualquier consideración especial.
    
    Args:
        arg1 (tipo): Descripción del primer argumento
        arg2 (tipo): Descripción del segundo argumento
        arg3 (tipo, optional): Descripción del argumento opcional. Default: valor_por_defecto
    
    Returns:
        tipo: Descripción de lo que retorna
    
    Raises:
        TipoError: Cuándo se lanza esta excepción
        ValueError: Cuándo se lanza esta otra excepción
    
    Examples:
        >>> resultado = nombre_funcion('entrada1', 'entrada2')
        >>> print(resultado)
    """
    # implementación
    return resultado
```

# U4.03 - Integración con APIs

## Uso de la Biblioteca requests, Llamada a la API y Conversión a DataFrame

In [ ]:
# Instalar requests
# %pip install requests

Attribution: <a href="https://www.exchangerate-api.com">Rates By Exchange Rate API</a>

# U4.04 - Pipeline ETL Completo

## Arquitectura General de un Pipeline ETL

El pipeline tiene 3 etapas:

1. **EXTRACT** (Extracción)
   - Leer datos de múltiples fuentes
   - CSV, Excel, APIs, bases de datos

2. **TRANSFORM** (Transformación)
   - Limpiar (espacios, tipos, duplicados)
   - Enriquecer (agregar columnas calculadas)
   - Normalizar (texto, categorías)

3. **LOAD** (Carga)
   - Guardar archivo procesado
   - Registrar en logs


Pero en la práctica moderna (también con Python) casi siempre se **descompone en más pasos** para que sea más controlable y confiable.

**ETL “clásico” (3 etapas)**

* **Extract:** leer de fuentes (CSV, Excel, API, BD)
* **Transform:** limpiar, unir, calcular, estandarizar
* **Load:** guardar en destino (CSV final, BD, data warehouse)

**Cómo se suele dividir hoy (muy común)**

Un pipeline real suele tener etapas adicionales como estas:

1. **Ingest / Extract**

   * extraer datos “tal cual” (raw)
   * a veces guardarlos primero (landing/raw)

2. **Validate / Quality checks**

   * columnas requeridas, tipos, nulos críticos, duplicados
   * reglas de negocio (ej: “precio > 0”)

3. **Transform / Clean / Enrich**

   * limpieza + cálculo de métricas + joins
   * normalización de texto, fechas, etc.

4. **Consolidate / Model**

   * dejar el dataset “listo para análisis” (schema final)
   * agregar particiones, columnas de auditoría, etc.

5. **Load / Publish**

   * cargar a destino: carpeta processed, BD, lake/warehouse
   * a veces en dos capas: “staging” → “final”

6. **Logging + Monitoring + Alerts** (transversal)

   * logs, métricas, notificaciones si falla

7. **Orchestration** (si escala)

   * programar, reintentos, dependencias (Airflow, Prefect, Dagster, etc.)

## Implementación del Pipeline ETL: Extracción, Transformación y Carga

Aquí vamos a reunir todas las etapas de nuestro Pipeline y además vamos agregar el manejo de excepciones.

In [ ]:
# %pip install openpyxl para importar documentos de excel


# U4.05 - Automatización básica

## Configuración de Logging, Ejecución del Script y Parametrización

Importante corregir las rutas, ya que hay algunas que funcionan en el notebook y otras que funcionan desde el .py

In [ ]:
"""
pipeline.py - Script ETL ejecutable desde terminal

Uso:
    python pipeline.py                    # Ejecutar con parámetros por defecto
    python pipeline.py --fecha 2024-02-15  # Ejecutar con fecha específica
    
Registra el flujo en: logs/pipeline.log
"""

Para ejecutar un script Python desde la terminal, primero asegúrate de que Python está instalado y en ejecutandose en el path del .py. Luego, simplemente usa:
```bash
python nombre_del_script.py [argumentos]
```